In [1]:
from __future__ import annotations

import pickle
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Any

import numpy as np
import os
import json
import gzip
import math

from stable_platform_matchings import Optimizer
from stable_platform_matchings.optimization.options import OptimizerParams, SolverOptions
from stable_platform_matchings.domain.instance import Instance
from stable_platform_matchings.graphs.road_graphs import RoadGraph
from pprint import pprint

In [2]:
n_id = 1

sim_size = 1

def reset_quantities():
    return {farmer.id: farmer.quantity for farmer in Platform.farmers}

def reset_fixed_costs():
    return {intermediary.id: Platform.dist_to_mill[intermediary.id]*4 for intermediary in Platform.intermediaries}

In [3]:
results = []

instance_str = "2020-08-27"
Platform = Instance.from_yaml('../data/anon_14_day_instances/'+instance_str+'.yaml')
Platform = Instance.from_yaml('../data/anon_14_day_instances/'+instance_str+'.yaml', force_quantities=reset_quantities())

rng = np.random.default_rng(1)

epsilon = {int.id: 2.0 for int in Platform.intermediaries}

with open("../data/graph_0-14960_00_new.pickle", 'rb') as pickle_file:
    G = pickle.load(pickle_file)

Platform.set_graph(RoadGraph(G))
farmer_quantities = {farmer.id: farmer.quantity for farmer in Platform.farmers}
het_costs = reset_fixed_costs()
parameters = {
    "epsilon":epsilon, 
    "solver": "gurobi", 
    "het_costs": het_costs,
}

pprint(het_costs)
pprint(farmer_quantities)

farmer_dist_to_mill = {farmer.id: farmer.dist_to_mill for farmer in Platform.farmers}
farmer_dirt_to_mill = {farmer.id: farmer.dirt_to_mill for farmer in Platform.farmers}
farmer_paved_to_mill = {farmer.id: farmer.paved_to_mill for farmer in Platform.farmers}

pprint(farmer_dirt_to_mill)
pprint(farmer_paved_to_mill)

{'beautiful_bohr': 251156.24797256076,
 'competent_mayer': 406399.4018487107,
 'elated_haslett': 977691.1445767505,
 'elegant_gagarin': 627180.0496084697,
 'elegant_mendel': 348573.77294422314,
 'exciting_fermi': 493149.4427404745,
 'gallant_cerf': 647257.1157146845,
 'hopeful_sanderson': 823884.0054250445,
 'keen_visvesvaraya': 1194805.381073437,
 'laughing_mestorf': 840198.7682361763,
 'loving_engelbart': 1054403.388237939,
 'peaceful_austin': 868601.4376568777,
 'quizzical_elgamal': 870359.6940502283,
 'vigorous_mccarthy': 672948.548382829}
{'beautiful_bohr_14_12': 2.0,
 'competent_mayer_14_1': 2.0,
 'elegant_gagarin_16_1': 0.8,
 'elegant_gagarin_98_1': 1.9,
 'elegant_mendel_23_8': 2.0,
 'elegant_mendel_54_6': 3.6,
 'elegant_mendel_69_3': 3.2,
 'gallant_cerf_11_5': 2.3,
 'gallant_cerf_12_12': 1.2,
 'gallant_cerf_19_2': 1.0,
 'gallant_cerf_29_4': 3.5,
 'laughing_mestorf_101_1': 0.5,
 'laughing_mestorf_11_8': 1.7,
 'laughing_mestorf_24_19': 2.8,
 'laughing_mestorf_62_1': 0.5,
 'loving

In [4]:
params = OptimizerParams(
    het_costs=het_costs,
    epsilons=epsilon,
    threads=14
)
options = SolverOptions(
    aggregate=True,
    stabilize_branch_extrema=False
)

opt = Optimizer(Platform, params)

summary_vanilla = opt.solve(options)

results.append({
    "instance_str": instance_str,
    "cost": het_costs,
    "epsilon": epsilon,
    "farmer_quantities": farmer_quantities,
    "summary_vanilla": summary_vanilla,
    "farmer_dirt_to_mill": farmer_dirt_to_mill,
    "famer_paved_to_mill": farmer_paved_to_mill,
})




============================= Optimizer Parameters =============================
---------------------------------- het_costs -----------------------------------
  {'beautiful_bohr': 251156.24797256076,
   'competent_mayer': 406399.4018487107,
   'elated_haslett': 977691.1445767505,
   'elegant_gagarin': 627180.0496084697,
   'elegant_mendel': 348573.77294422314,
   'exciting_fermi': 493149.4427404745,
   'gallant_cerf': 647257.1157146845,
   'hopeful_sanderson': 823884.0054250445,
   'keen_visvesvaraya': 1194805.381073437,
   'laughing_mestorf': 840198.7682361763,
   'loving_engelbart': 1054403.388237939,
   'peaceful_austin': 868601.4376568777,
   'quizzical_elgamal': 870359.6940502283,
   'vigorous_mccarthy': 672948.548382829}
----------------------------------- epsilons -----------------------------------
  {'beautiful_bohr': 2.0,
   'competent_mayer': 2.0,
   'elated_haslett': 2.0,
   'elegant_gagarin': 2.0,
   'elegant_mendel': 2.0,
   'exciting_fermi': 2.0,
   'gallant_cerf': 

In [5]:
pprint(results[0])

{'cost': {'beautiful_bohr': 251156.24797256076,
          'competent_mayer': 406399.4018487107,
          'elated_haslett': 977691.1445767505,
          'elegant_gagarin': 627180.0496084697,
          'elegant_mendel': 348573.77294422314,
          'exciting_fermi': 493149.4427404745,
          'gallant_cerf': 647257.1157146845,
          'hopeful_sanderson': 823884.0054250445,
          'keen_visvesvaraya': 1194805.381073437,
          'laughing_mestorf': 840198.7682361763,
          'loving_engelbart': 1054403.388237939,
          'peaceful_austin': 868601.4376568777,
          'quizzical_elgamal': 870359.6940502283,
          'vigorous_mccarthy': 672948.548382829},
 'epsilon': {'beautiful_bohr': 2.0,
             'competent_mayer': 2.0,
             'elated_haslett': 2.0,
             'elegant_gagarin': 2.0,
             'elegant_mendel': 2.0,
             'exciting_fermi': 2.0,
             'gallant_cerf': 2.0,
             'hopeful_sanderson': 2.0,
             'keen_visvesvaraya'

In [ ]:
params = OptimizerParams(
    het_costs=het_costs,
    epsilons=epsilon,
    threads=14
)
options = SolverOptions(
    aggregate=True
)

opt = Optimizer(Platform, params)

summary_vanilla = opt.solve(options)

results.append({
    "instance_str": instance_str,
    "cost": het_costs,
    "epsilon": epsilon,
    "farmer_quantities": farmer_quantities,
    "summary_vanilla": summary_vanilla,
    "farmer_dirt_to_mill": farmer_dirt_to_mill,
    "famer_paved_to_mill": farmer_paved_to_mill,
})




============================= Optimizer Parameters =============================
---------------------------------- het_costs -----------------------------------
  {'beautiful_bohr': 251156.2479725607,
   'competent_mayer': 406399.4018487107,
   'elated_haslett': 977691.1445767505,
   'elegant_gagarin': 627180.0496084698,
   'elegant_mendel': 348573.77294422314,
   'exciting_fermi': 493149.4427404745,
   'gallant_cerf': 647257.1157146845,
   'hopeful_sanderson': 823884.0054250445,
   'keen_visvesvaraya': 1194805.3810734372,
   'laughing_mestorf': 840198.7682361763,
   'loving_engelbart': 1054403.388237939,
   'peaceful_austin': 868601.4376568777,
   'quizzical_elgamal': 870359.6940502283,
   'vigorous_mccarthy': 672948.548382829}
----------------------------------- epsilons -----------------------------------
  {'beautiful_bohr': 2.0,
   'competent_mayer': 2.0,
   'elated_haslett': 2.0,
   'elegant_gagarin': 2.0,
   'elegant_mendel': 2.0,
   'exciting_fermi': 2.0,
   'gallant_cerf': 

In [ ]:
pprint(results[0])

{'cost': {'beautiful_bohr': 251156.2479725607,
          'competent_mayer': 406399.4018487107,
          'elated_haslett': 977691.1445767505,
          'elegant_gagarin': 627180.0496084698,
          'elegant_mendel': 348573.77294422314,
          'exciting_fermi': 493149.4427404745,
          'gallant_cerf': 647257.1157146845,
          'hopeful_sanderson': 823884.0054250445,
          'keen_visvesvaraya': 1194805.3810734372,
          'laughing_mestorf': 840198.7682361763,
          'loving_engelbart': 1054403.388237939,
          'peaceful_austin': 868601.4376568777,
          'quizzical_elgamal': 870359.6940502283,
          'vigorous_mccarthy': 672948.548382829},
 'epsilon': {'beautiful_bohr': 2.0,
             'competent_mayer': 2.0,
             'elated_haslett': 2.0,
             'elegant_gagarin': 2.0,
             'elegant_mendel': 2.0,
             'exciting_fermi': 2.0,
             'gallant_cerf': 2.0,
             'hopeful_sanderson': 2.0,
             'keen_visvesvaraya'